In [1]:
import transformers
import time
import torch
import langchain_community
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoProcessor, AutoModelForMultimodalLM
from bert_score import BERTScorer
from nltk.translate.meteor_score import meteor_score
import pprint
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
import pypdf
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore 
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma

/Users/danclancy/Desktop/Dissertation Folder/RAG LLM Code Area/Dissertation_Repository/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/0f/9853n3y529v8dtlr3sfn_zf00000gn/T/ipykernel_65883/1135562601.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community


In [2]:

model_name = "google/gemma-4-E4B-it"
quantization_config = BitsAndBytesConfig(load_in_4bit=True,
                                          bnb_4bit_compute_dtype=torch.bfloat16,
                                          
                                          bnb_4bit_quant_type="nf4")

processor = AutoProcessor.from_pretrained(model_name)



model = AutoModelForMultimodalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="mps",
    
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights: 100%|██████████| 2076/2076 [00:00<00:00, 2865.54it/s]


In [3]:
#embedding section
embeddingsTWO = HuggingFaceEmbeddings(
    model_name="google/embeddinggemma-300m",
    encode_kwargs={"normalize_embeddings": True},
)

Loading weights: 100%|██████████| 314/314 [00:00<00:00, 6962.79it/s]


In [4]:
vector_store = Chroma(
    collection_name="LocalEmbeddingsGemma300m",
    embedding_function=embeddingsTWO,
    persist_directory="./chroma_RAG_db"
)

In [ ]:
# splitter = RecursiveCharacterTextSplitter(
#     chunk_size = 1000, chunk_overlap=200, add_start_index= True
# )

In [2]:
# loader = PyPDFLoader("/Users/danclancy/Desktop/Dissertation Folder/Astrometry and geodesy with radio interferometry- experiments, models, results.pdf")
# document = loader.load()
# docSplit = splitter.split_documents(document)
# vector_store.add_documents(docSplit)



In [1]:
# #Context = "'This is a white hole, the time reverse of a black hole. Both black and white holes are allowed by G.R. because of the time reversibility of Einstein’s equations, but white holes require very special initial conditions near the singularity, whereas black holes do not, so only black holes can occur in practice ', 'The outgoing Eddington-Finkelstein spacetime has properties which are opposite to those of the black hole. Naturally, one says that these properties describe a white hole. The chronological future of an event for which r > rs is contained in the region r > rs. This means that not even light can enter the region r < rs, which is called the interior of the white hole. The worldline of a particle that starts with r < rs either leaves the region r < rs or approaches r = rs. This means that everything that is in the interior of the white hole tends to leave it. In contrast with black holes, there is no evidence that white holes exist in nature.','The velocity β is commonly called the shift in the ADM formalism26,27, but in this paper we refer to β as the river velocity. The river velocity β is positive for a black hole (infalling), negative for a white hole (outfalling). Horizons occur where the river velocity β equals the speed of light,β = ±1, with β = 1 for black hole horizons, and β = −1 for white hole horizons.','The inflow velocity β hits the speed of light at the outer horizon r+, reaches a maximum velocity between the outer and inner horizons, slows back down to the speed of light at the inner horizon r−, and slows all the way to zero velocity at the turnaround radius r0 = Q2 2M . (22) At this point the flow of space turns around, accelerates back outward through another inner horizon, the socalled Cauchy horizon, into a white hole, and bursts through the outer horizon of the white hole into a new universe. Sadly, the RN solution is not realistic, and its promise of passages to other universes is moot. The RN solution describes the geometry of empty space surrounding a classical point charge of infinitesimal size, and as such consists of a gravitationally repulsive singularity of infinite negative mass sheathed in an electric field containing an infinite positive mass balanced so as to yield a finite mass M at infinity. See40 for an entry to the literature on the intriguing subject of what really happens inside charged black holes.','They have shown that loop quantum gravity—tentative theory of quantum gravity— predicts that spacetime continues across the center of the hole into a new region that has the geometry of the interior of a white hole, and is located in the future of the black hole (see Fig 1). A white hole is the time-reversed image of a black hole: in it, matter can only move outwards. The passage “across the center” into a future region is counterintuitive; it is possible thanks to the strong distortion of the spacetime geometry inside the hole allowed by general relativity.','White holes have received far less attention from researchers than black holes (for a review, see, e.g., [1]). This is understandable, given that conditions in our universe readily lead to black hole formation, whereas white hole creation has neither been observed nor is expected to have occurred in the history of the universe. However, white holes are themselves fundamental objects and worthy of further study. White holes are time-reversed black holes, and therefore characterized by the same quantum numbers: mass, angular momentum, charge. While a classical black hole spacetime has a singularity in the future, a white hole has one in the past. If quantum gravitational effects can resolve black hole singularities, then white holes need not result from singular initial conditions. (In any case the initial white hole singularity is not directly visible to observers.)','Standard quantum mechanical reasoning suggests that any initial state which evolves into a black hole also has some nonzero probability to evolve into a white hole. Note we are referring to a quantum gravitational process, and are making the assumption that even in quantum gravity tunneling between two states with the same quantum numbers has non-zero (although possibly very small) probability. For example, it is known that the collision of two sufficiently energetic particles can create a black hole [2]. Because the quantum numbers are the same we would expect that the same energetic particles have a small but non-zero probability of producing a white hole with the same quantum numbers [3]. Since large black holes are long-lived, there are some white hole states (corresponding to time slices late in the black hole’s existence) that persist for a long time before exploding. Thus, long-lived white holes are a consequence of quantum mechanics and the properties of black holes.'"
# messages = [
#     {"role": "system", "content": f"Role: You are the world's foremost black hole expert. Explain simply like im 5"},


#     {"role": "user", "content": "Explain further the previous white hole description"}
# ]
# start = time.time()

# #input_ids = tokenizer(prompt, return_tensors="pt").to(model.device)

# input_ids = processor.apply_chat_template(
#     messages,
#     tokenize = False,
#     add_generation_prompt=True,
#     enable_thinking=False)

# inputs= processor(text=input_ids, return_tensors="pt").to(model.device)
# input_len = inputs["input_ids"].shape[-1]
# outputs = model.generate(**inputs, max_new_tokens=256)
# end = time.time()
# response = processor.decode(outputs[0][input_len:], skip_special_tokens=True)
# response = response.replace("\n"," ")
# response = response.replace("#"," ")
# response = response.replace("*","")
# pprint.pp(processor.parse_response(response))

# length= end - start
# print(f"It took {length} seconds to run")

In [5]:
import gradio as gr

In [6]:
from pylatexenc.latex2text import LatexNodes2Text

In [7]:
latex = r"$r_s = a \cdot M$"
print(LatexNodes2Text().latex_to_text(latex))

r_s = a · M


In [ ]:
# RAG full prompt template with RAG formula in there.
# "Role: You are a physics expert giving a tour of your lab to a group of visitors. These visitors are all high school age and above. You keep this in mind when you are asked the user query.Task: You are to answer the user query in a complete manner. You must adapt the complexity of your answer to the complexity of the question that is asked. Tonality and Format: You are to answer in an informal friendly tone and answer with length that is as long as you see fit but no longer that a large paragraph. Context: {'\n'.join([f'-{chunk}'for chunk in results])}Constraints: You cannot by any means make up any new information. You will always tell the respondents that you do not know something if that is the case. You are only to use information given in the Context to answer the user query and no other information. YOU CAN ONLY USE INFORMATION FROM THE CONTEXT TO ANSWER THE USER QUESTION. Do not give the user any redundant information.

In [ ]:
# Feynman RAG full template 
# "Role:You are  a Physics University Tutor specializing in black holes. You are helping a group of first year university students answer a set of questions that they have been given by their lecturer. You are tasked with answering the user query. Task: There are 20 questions in total that they will need your help answering and you must actively assist them with all 20 questions. When asked a query you are to give them short and concise answers that still answers the query completely. Tonality and Format: You are to respond to the students in a friendly but professional way using clear words that are non-academic in nature and simple to understand. Context: {'\n'.join([f'-{chunk}'for chunk in results])} Constraints: Answer the questions clearly and effectively as stated above. You are only to use information given in the Context to answer the question and no other information. YOU CAN ONLY USE INFORMATION FROM THE CONTEXT TO ANSWER THE QUESTION Do not give the students any information that is redundant and that they do not need. You will always tell the respondents that you do not know something if that is the case.  Before answering the question you must do the following 4 step process ONLY ONCE. 1.You must explain your response in simple terms as if explaining it to a child. 2. You must identify gaps in your explanation or any inconsistencies in your explanation. 3. You must then re-write and refine your explanation based on these gaps or inconsistencies. 4. You must then produce the answer in a concise manner.

In [11]:
def QA (message, history):
    results = vector_store.similarity_search(message)

    messages = history + [{"role": "system", "content": f"""Role: You are a physics expert giving a tour of your lab to a group of visitors. These visitors are all high school age and above. You keep this in mind when you are asked the user query.
    
    Task: You are to answer the user query in a complete manner. You must adapt the complexity of your answer to the complexity of the question that is asked. 
    
    Tonality and Format: You are to answer in an informal friendly tone and answer with length that is as long as you see fit but no longer that a large paragraph. 
    
    Context: {'\n'.join([f'-{chunk}'for chunk in results])}
    
    Constraints: You cannot by any means make up any new information. You will always tell the respondents that you do not know something if that is the case. You are only to use information given in the Context to answer the user query and no other information. YOU CAN ONLY USE INFORMATION FROM THE CONTEXT TO ANSWER THE USER QUESTION. Do not give the user any redundant information. You must always greet the respondents with Hello.""" },{"role": "user", "content": message}]
    input_ids = processor.apply_chat_template(messages, tokenize = False, add_generation_prompt=True, enable_thinking=False)
    inputs= processor(text=input_ids, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]
    outputs = model.generate(**inputs, max_new_tokens=1024)
    response = processor.decode(outputs[0][input_len:], skip_special_tokens=True)
    responselatex = LatexNodes2Text().latex_to_text(response)
    return responselatex
    
    

In [12]:
demo = gr.ChatInterface(QA, api_name="chat")

demo.launch(theme=gr.themes.Citrus())

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
